# Day2 - Bronze: Exchange Rates

**Objetivo:** Criar a tabela Bronze `exchange_rates` a partir dos arquivos JSON salvos na camada Landing e carregar os dados com idempotência.

| Item | Valor |
|---|---|
| **Camada** | Bronze |
| **Entrada** | `/Volumes/{catalog}/landing/files/exchangerates/*.json` |
| **Saída** | `{catalog}.bronze.exchange_rates` |
| **Ordem no pipeline** | Após `Day1-Ingestao-Exchange-Rates` (Landing) |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Criação da tabela Bronze

Define o esquema da tabela `exchange_rates` com `CREATE OR REPLACE TABLE`, garantindo idempotência na reexecução.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`bronze`.`exchange_rates` (
    base STRING,
    data_base DATE,
    moeda STRING,
    valor DOUBLE,
    updated TIMESTAMP
)
USING DELTA
    COMMENT "Exchange rates of the world";

## Carga dos dados

Lê todos os arquivos JSON da Landing, faz o explode do mapa de moedas e insere na tabela Bronze. O `TRUNCATE` antes do `INSERT` garante idempotência — reexecutar não duplica linhas.

In [0]:
%sql
TRUNCATE TABLE `${catalog}`.`bronze`.`exchange_rates`;

INSERT INTO `${catalog}`.`bronze`.`exchange_rates`
    SELECT
        base,
        data_base,
        moeda,
        valor,
        from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS updated
    FROM (
        SELECT
            from_utc_timestamp(from_unixtime(timestamp), 'America/Sao_Paulo') AS data_extracao,
            base,
            TO_DATE(date) AS data_base,
            rates
        FROM read_files(
            '/Volumes/${catalog}/landing/files/exchangerates/*.json',
            format => 'json',
            multiLine => true)
    ) 
    LATERAL VIEW EXPLODE(FROM_JSON(TO_JSON(rates), 'MAP<STRING, DOUBLE>')) AS moeda, valor;